# Inspect Silver Pageview Data

Use this notebook to inspect the local Silver layer under `data/processed/silver/pageviews`. It checks partition coverage, schema, sampled Parquet rows, basic data quality rules, quarantine outputs, and optional duplicate-key checks.

The notebook is read-only. Expensive full scans are disabled by default.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from wikitrend.pageviews import DEFAULT_SOURCE_PROJECTS, PROJECT_CODE_MAP

SILVER_DIR = ROOT / "data" / "processed" / "silver" / "pageviews"
QUARANTINE_DIR = ROOT / "data" / "processed" / "quarantine" / "pageviews"
REJECTION_SUMMARY_DIR = ROOT / "data" / "processed" / "quarantine" / "pageviews_rejection_summary"
MANIFEST_PATH = ROOT / "data" / "raw" / "pageviews_manifest.json"

SILVER_DIR, QUARANTINE_DIR, MANIFEST_PATH

## 2. Parameters

In [ ]:
SAMPLE_FILES = 8
SAMPLE_ROWS_PER_FILE = 2_000

# Turn these on only when you want complete validation scans.
COUNT_ALL_ROWS = False
CHECK_DUPLICATE_NATURAL_KEYS = False

# Natural key expected to be unique within Silver page-hour facts.
NATURAL_KEY = ["date", "hour", "source_project", "page_title"]

## 3. Bronze Manifest Context

In [ ]:
assert SILVER_DIR.exists(), f"Missing Silver directory: {SILVER_DIR}"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}
manifest_df = pd.DataFrame(manifest.get("files", []))
if not manifest_df.empty:
    manifest_df["timestamp_hour"] = pd.to_datetime(manifest_df["timestamp_hour"], utc=True)

manifest_summary = {
    "plan_id": manifest.get("plan_id"),
    "bronze_manifest_files": len(manifest_df),
    "first_bronze_hour": manifest_df["timestamp_hour"].min() if not manifest_df.empty else None,
    "last_bronze_hour": manifest_df["timestamp_hour"].max() if not manifest_df.empty else None,
}

manifest_summary

## 4. Silver Partition Inventory

In [ ]:
PARTITION_RE = re.compile(
    r"date=(?P<date>[^/\\]+)[/\\]hour=(?P<hour>[^/\\]+)[/\\]"
    r"project=(?P<project>[^/\\]+)[/\\]access_mode=(?P<access_mode>[^/\\]+)"
)


def parse_partition(path: Path) -> dict[str, object]:
    relative = path.relative_to(SILVER_DIR).as_posix()
    match = PARTITION_RE.search(relative)
    if not match:
        return {"date": None, "hour": None, "project": None, "access_mode": None}
    payload = match.groupdict()
    payload["hour"] = int(payload["hour"])
    return payload


silver_files = sorted(path for path in SILVER_DIR.rglob("*.parquet") if path.is_file())
silver_file_df = pd.DataFrame(
    [
        {
            "path": path,
            "relative_path": path.relative_to(SILVER_DIR).as_posix(),
            "size_bytes": path.stat().st_size,
            **parse_partition(path),
        }
        for path in silver_files
    ]
)

inventory_summary = {
    "parquet_files": len(silver_file_df),
    "size_gib": round(silver_file_df["size_bytes"].sum() / 1024**3, 2) if not silver_file_df.empty else 0,
    "dates": int(silver_file_df["date"].nunique()) if not silver_file_df.empty else 0,
    "hours": int(silver_file_df[["date", "hour"]].drop_duplicates().shape[0]) if not silver_file_df.empty else 0,
    "partition_combinations": int(silver_file_df[["date", "hour", "project", "access_mode"]].drop_duplicates().shape[0]) if not silver_file_df.empty else 0,
}

inventory_summary

In [ ]:
silver_file_df.head()

## 5. Coverage Checks

In [ ]:
expected_dimension_rows = []
for source_project in DEFAULT_SOURCE_PROJECTS:
    dimensions = PROJECT_CODE_MAP[source_project]
    expected_dimension_rows.append(
        {
            "source_project": source_project,
            "project": dimensions.project,
            "access_mode": dimensions.access_mode,
        }
    )
expected_dimensions_df = pd.DataFrame(expected_dimension_rows).drop_duplicates()

hourly_partitions = silver_file_df[["date", "hour", "project", "access_mode"]].drop_duplicates()
hour_coverage = (
    hourly_partitions.groupby(["date", "hour"], dropna=False)
    .agg(partition_combinations=("project", "size"))
    .reset_index()
    .sort_values(["date", "hour"])
)

expected_combinations_per_hour = len(expected_dimensions_df)
coverage_issues = hour_coverage.loc[
    hour_coverage["partition_combinations"] != expected_combinations_per_hour
]

{
    "expected_project_access_combinations_per_hour": expected_combinations_per_hour,
    "hours_with_any_silver_partition": len(hour_coverage),
    "hours_with_unexpected_partition_count": len(coverage_issues),
}

In [ ]:
coverage_issues.head(50)

## 6. Schema

In [ ]:
silver_dataset = ds.dataset(SILVER_DIR, format="parquet", partitioning="hive")
silver_dataset.schema

## 7. Sample Rows

In [ ]:
def read_sample_files(files: list[Path], rows_per_file: int) -> pd.DataFrame:
    frames = []
    for path in files:
        table = pq.read_table(path)
        frame = table.slice(0, rows_per_file).to_pandas()
        partitions = parse_partition(path)
        for name, value in partitions.items():
            if name not in frame.columns:
                frame[name] = value
        frame["source_parquet_file"] = path.relative_to(SILVER_DIR).as_posix()
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


sample_files = silver_files[:SAMPLE_FILES]
sample_df = read_sample_files(sample_files, SAMPLE_ROWS_PER_FILE)

sample_df.head()

## 8. Sample Data Quality Checks

In [ ]:
if sample_df.empty:
    print("No sample rows loaded.")
else:
    quality_summary = {
        "sample_rows": len(sample_df),
        "null_page_titles": int(sample_df["page_title"].isna().sum()),
        "null_normalized_titles": int(sample_df["normalized_title"].isna().sum()) if "normalized_title" in sample_df else None,
        "negative_view_count": int((sample_df["view_count"] < 0).sum()),
        "negative_response_size": int((sample_df["response_size"] < 0).sum()),
        "projects": sorted(sample_df["project"].dropna().unique().tolist()),
        "access_modes": sorted(sample_df["access_mode"].dropna().unique().tolist()),
    }
    quality_summary

In [ ]:
if not sample_df.empty:
    display(
        sample_df.groupby(["date", "hour", "project", "access_mode"], dropna=False)
        .agg(rows=("page_title", "size"), views=("view_count", "sum"), response_size=("response_size", "sum"))
        .sort_values("views", ascending=False)
        .head(50)
    )

In [ ]:
if not sample_df.empty:
    display(
        sample_df.sort_values("view_count", ascending=False)[
            ["date", "hour", "project", "access_mode", "normalized_title", "view_count", "response_size"]
        ].head(25)
    )

## 9. Quarantine Inventory

In [ ]:
quarantine_files = sorted(QUARANTINE_DIR.rglob("*.parquet")) if QUARANTINE_DIR.exists() else []
rejection_summary_files = (
    sorted(REJECTION_SUMMARY_DIR.rglob("*.parquet")) if REJECTION_SUMMARY_DIR.exists() else []
)

quarantine_summary = {
    "quarantine_dir_exists": QUARANTINE_DIR.exists(),
    "quarantine_parquet_files": len(quarantine_files),
    "quarantine_size_mib": round(sum(path.stat().st_size for path in quarantine_files) / 1024**2, 2),
    "rejection_summary_dir_exists": REJECTION_SUMMARY_DIR.exists(),
    "rejection_summary_parquet_files": len(rejection_summary_files),
}

quarantine_summary

In [ ]:
if quarantine_files:
    quarantine_sample_df = read_sample_files(quarantine_files[: min(SAMPLE_FILES, len(quarantine_files))], 500)
    display(quarantine_sample_df.head())
    if "invalid_reason" in quarantine_sample_df:
        display(quarantine_sample_df["invalid_reason"].value_counts())
else:
    print("No quarantine Parquet files found.")

## 10. Optional Full Row Count

In [ ]:
if COUNT_ALL_ROWS:
    total_rows = silver_dataset.count_rows()
    print(f"Silver rows: {total_rows:,}")
else:
    print("Set COUNT_ALL_ROWS = True to count all Silver rows.")

## 11. Optional Duplicate Natural Key Check

This check scans the full dataset into grouped keys. It can be expensive on large Silver outputs.

In [ ]:
if CHECK_DUPLICATE_NATURAL_KEYS:
    key_table = silver_dataset.to_table(columns=NATURAL_KEY)
    key_df = key_table.to_pandas()
    duplicate_keys = (
        key_df.value_counts(NATURAL_KEY)
        .rename("count")
        .reset_index()
        .query("count > 1")
        .sort_values("count", ascending=False)
    )
    display(duplicate_keys.head(50))
    print(f"Duplicate key groups: {len(duplicate_keys):,}")
else:
    print("Set CHECK_DUPLICATE_NATURAL_KEYS = True to scan duplicate natural keys.")

## 12. Optional Spark Inspection

Use this only in an environment with PySpark installed and configured.

In [ ]:
USE_SPARK = False

if USE_SPARK:
    from wikitrend.silver import create_spark_session
    from wikitrend.storage import ensure_spark_path

    spark = create_spark_session("WikiTrend Silver Inspection")
    silver_spark_df = spark.read.parquet(ensure_spark_path(SILVER_DIR))
    silver_spark_df.printSchema()
    display(silver_spark_df.groupBy("date", "project", "access_mode").count().orderBy("date", "project", "access_mode").limit(50).toPandas())
else:
    print("Set USE_SPARK = True to run Spark-based inspection.")

## 13. What To Look For

- Silver partition hours should match the Bronze manifest hours.
- Each hour should usually have the expected project/access partitions.
- Schema should include date/hour, source project, canonical project dimensions, title fields, counts, and source filename.
- Sample quality checks should show no null titles and no negative metrics.
- Quarantine rows are expected for unsupported source projects or malformed lines; inspect their reasons before treating them as errors.
- Run the optional duplicate-key check only when you need a full validation pass.